# 03 — DCGAN on MNIST (PyTorch)

PyTorch-only GAN implementation. Dataset is downloaded by torchvision.

## 1. Data and device

In [ ]:
import torch, torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
import matplotlib.pyplot as plt
print(torch.__version__,torchvision.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds=datasets.MNIST('data',train=True,download=True,transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.5,),(.5,))]))
loader=DataLoader(ds,batch_size=128,shuffle=True,drop_last=True)

## 2. Generator and discriminator

In [ ]:
Z=100
class Generator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.ConvTranspose2d(Z,256,7,1,0,bias=False),nn.BatchNorm2d(256),nn.ReLU(True),nn.ConvTranspose2d(256,128,4,2,1,bias=False),nn.BatchNorm2d(128),nn.ReLU(True),nn.ConvTranspose2d(128,1,4,2,1,bias=False),nn.Tanh())
 def forward(self,z): return self.net(z)
class Discriminator(nn.Module):
 def __init__(self):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(64,128,4,2,1,bias=False),nn.BatchNorm2d(128),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(128*7*7,1))
 def forward(self,x): return self.net(x)
G=Generator().to(device); D=Discriminator().to(device); optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999)); optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999)); bce=nn.BCEWithLogitsLoss()

## 3. Adversarial training step

In [ ]:
fixed_z=torch.randn(16,Z,1,1,device=device)for epoch in range(5): for real,_ in loader:  real=real.to(device); n=len(real); ones=torch.ones(n,1,device=device); zeros=torch.zeros(n,1,device=device)  # Update discriminator  optD.zero_grad(set_to_none=True); z=torch.randn(n,Z,1,1,device=device); fake=G(z)  dloss=bce(D(real),ones)+bce(D(fake.detach()),zeros); dloss.backward(); optD.step()  # Update generator  optG.zero_grad(set_to_none=True); z_g=torch.randn(n,Z,1,1,device=device); fake_for_g=G(z_g); gloss=bce(D(fake_for_g),ones); gloss.backward(); optG.step() print('epoch',epoch+1,'D',float(dloss),'G',float(gloss)) with torch.inference_mode(): samples=G(fixed_z).cpu().squeeze(1) fig,ax=plt.subplots(4,4,figsize=(4,4)) for i,a in enumerate(ax.flat): a.imshow(samples[i],cmap='gray'); a.axis('off') plt.show()

## GAN tips

Track fixed-noise outputs, diversity, loss trends and checkpoint state. GAN losses do not directly measure image quality. Try label smoothing and spectral normalization as separate controlled experiments.